#Ingest sprints.json files from sprints folder
1. Read the file using Spark DataFrame reader API
2. Define and enforce Schema
3. Add Metadata Columns
    - Source File
    - Ingestion TimeStamp
4. Write to bronze delta table

Note: Here JSON is in multi-line Format

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f"{landing_folder_path}/{v_batch_id}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
source_path

### Step 1& 2- Read the json file using the DataFrame Reader API and enforce the schema

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DoubleType,DateType,IntegerType,FloatType
                
                                            
sprints_schema = StructType([
            StructField('date',DateType()),
            StructField('raceName',StringType()),
            StructField('round',IntegerType()),
            StructField('season',IntegerType()),
            StructField('url',StringType()),
            StructField('constructorId',StringType()),
            StructField('driverId',StringType()),  
            StructField('grid',IntegerType()),
            StructField('laps',IntegerType()),
            StructField('number',IntegerType()),
            StructField('points',FloatType()),
            StructField('position',IntegerType()),
            StructField('positionText',StringType()),
            StructField('status',StringType()),                                  ])



In [0]:
sprints_df = (
    spark.read
         .format("json")
         .option("header","true")
#        .option("inferSchema","true")
         .schema(sprints_schema)
         .option("multiline","true")
         .option("mode","FAILFAST")
         .load(source_path)
        )


In [0]:
display(sprints_df)

###Step -3 Add Metadata Columns
- Source File
- Ingestion TimeStamp



In [0]:

sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
display(sprints_final_df)

### Step 3- Write to bronze delta table

In [0]:
v_batch_id

In [0]:
write_to_bronze(
    input_df = sprints_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
Select season,count(*)
FROM formula1.bronze.sprints
GROUP BY season
Order BY season;